# Activity search demo
This notebook demonstrates how to use the `UniprotInterface` class from the `bioseq_dl` package tto get target activity information from Uniprot.


In [1]:
import pandas as pd

from bioseq_dl import UniprotInterface

### Define the query and fields
- Fields: List of entry sections to be returned. More fields can be found in the [Uniprot documentation](https://rest.uniprot.org/configure/uniprotkb/result-fields).
- Sort: Specify field by wich to sort results.

In [2]:
activities = [
    "antibacterial",
    "antifungal",
    "antiviral",
    "antitumor",
    "antidiabetic",
    "antiinflammatory",
    "antioxidative",
    "immunomodulating",
    "cytotoxic",
    "protease-inhibitor",
]
fields = [
    "accession",
    "protein_name",
    "sequence",
    "ec",
    "lineage",
    "organism_name",
    "xref_pfam",
    "xref_alphafolddb",
    "xref_pdb",
    "go_id",
]
sort = ["accession asc"]

### Instantiate the API

In [3]:
instance = UniprotInterface()

### Submitting the query
This step will take a few seconds to complete.

In [19]:
results = {}
for activity in activities:
    print(f"Searching for activity: {activity}")
    response, _ = instance.submit_stream(
        query=f"{activity} AND reviewed:true", fields=fields, sort=sort, include_isoform=False
    )

    results[activity] = response

Searching for activity: antibacterial
Searching for activity: antifungal
Searching for activity: antiviral
Searching for activity: antitumor
Searching for activity: antidiabetic
Searching for activity: antiinflammatory
Searching for activity: antioxidative
Searching for activity: immunomodulating
Searching for activity: cytotoxic
Searching for activity: protease-inhibitor


### Parsing results
After getting the results you can parse them as a pandas DataFrame using the `parse` method.
- query: The same query used to get the results. A good practice is to tag how the results were obtained (e.g., "antibacterial AND reviewed:true").
- results: The results object obtained from the `get_stream_response` method.
- extract_fields: List of fields to extract from the response. If `None`, all fields will be extracted.

For example, to get the first 5 entries with antibacterial activity:

In [26]:
parsed, _ = instance.parse(
    results=results["antibacterial"],
    extract_fields=None,
    format="dataframe",
)
parsed.head(5)

,accession,protein_name,ec,organism_name,organism_id,lineage,sequence,length,alphafold_ids,biogrid_ids,...,kegg_ids,panther_ids,pathwaycommons_ids,pdb_ids,pfam_ids,pride_ids,reactome_ids,refseq_ids,sabiork_ids,string_ids
0,A0A023PMT2,Cecropin-B1,None,Aedes aegypti,7159,"[Eukaryota, Metazoa, Ecdysozoa, Arthropoda, He...",MNFSKVFALVLLIGLVLLTGHTEAGGLKKLGKKLEGVGKRVFKASE...,60,[Q17NR2],[],...,[],[],[],[],[PF00272],[],[],[],[],[]
1,A0A024BTN9,L-amino acid oxidase Bs29,None,Bothriechis schlegelii,44725,"[Eukaryota, Metazoa, Chordata, Craniata, Verte...",SCADDRNPLEECFQETDYEEFLEIARNGLKATSNPKHVVIVGAGMS...,498,[A0A024BTN9],[],...,[],[],[],[],[PF01593],[],[],[],[],[]
2,A0A076FF10,"Flavonoid 8-hydroxylase 2, chloroplastic",[1.14.15.-],Ocimum basilicum,39350,"[Eukaryota, Viridiplantae, Streptophyta, Embry...",MEVLQASSLSFQLLRRHSRNNLINKFRNPSLPRIHMPRQNIDLKTF...,519,[A0A076FF10],[],...,[],[],[],[],"[PF08417, PF00355]",[],[],[],[],[]
3,A0A076FFM5,"Flavonoid 8-hydroxylase 1, chloroplastic",[1.14.15.-],Ocimum basilicum,39350,"[Eukaryota, Viridiplantae, Streptophyta, Embry...",MPFPMEVLQASSLSFPLLRRHSRNNLINKFRNPTLPRIDIPRQNID...,523,[A0A076FFM5],[],...,[],[],[],[],"[PF08417, PF00355]",[],[],[],[],[]
4,A0A084G2W8,Fungal defensin scedosporisin-2,None,Pseudallescheria apiosperma,563466,"[Eukaryota, Fungi, Dikarya, Ascomycota, Pezizo...",MKFSNISIAALFTILASTAMAAPAADSPDSIVAREPAPVEETYEAP...,94,[A0A084G2W8],[],...,[],[],[],[],[PF01097],[],[],[],[],[]


### Parsing and merging results

In [27]:
merged_df = pd.DataFrame()
for activity in activities:
    print(f"Parsing results for activity: {activity}")
    df, _ = instance.parse(
        results=results[activity],
        extract_fields=None,
        format="dataframe",
    )
    print(f"Number of entries found for {activity}: {len(df)}")
    merged_df = pd.concat([merged_df, df], ignore_index=True)

Parsing results for activity: antibacterial
Number of entries found for antibacterial: 2379
Parsing results for activity: antifungal
Number of entries found for antifungal: 1215
Parsing results for activity: antiviral
Number of entries found for antiviral: 3940
Parsing results for activity: antitumor
Number of entries found for antitumor: 481
Parsing results for activity: antidiabetic
Number of entries found for antidiabetic: 47
Parsing results for activity: antiinflammatory
Number of entries found for antiinflammatory: 27
Parsing results for activity: antioxidative
Number of entries found for antioxidative: 55
Parsing results for activity: immunomodulating
Number of entries found for immunomodulating: 11
Parsing results for activity: cytotoxic
Number of entries found for cytotoxic: 1730
Parsing results for activity: protease-inhibitor
Number of entries found for protease-inhibitor: 3272


In [28]:
merged_df

,accession,protein_name,ec,organism_name,organism_id,lineage,sequence,length,alphafold_ids,biogrid_ids,...,kegg_ids,panther_ids,pathwaycommons_ids,pdb_ids,pfam_ids,pride_ids,reactome_ids,refseq_ids,sabiork_ids,string_ids
0,A0A023PMT2,Cecropin-B1,None,Aedes aegypti,7159,"[Eukaryota, Metazoa, Ecdysozoa, Arthropoda, He...",MNFSKVFALVLLIGLVLLTGHTEAGGLKKLGKKLEGVGKRVFKASE...,60,[Q17NR2],[],...,[],[],[],[],[PF00272],[],[],[],[],[]
1,A0A024BTN9,L-amino acid oxidase Bs29,None,Bothriechis schlegelii,44725,"[Eukaryota, Metazoa, Chordata, Craniata, Verte...",SCADDRNPLEECFQETDYEEFLEIARNGLKATSNPKHVVIVGAGMS...,498,[A0A024BTN9],[],...,[],[],[],[],[PF01593],[],[],[],[],[]
2,A0A076FF10,"Flavonoid 8-hydroxylase 2, chloroplastic",[1.14.15.-],Ocimum basilicum,39350,"[Eukaryota, Viridiplantae, Streptophyta, Embry...",MEVLQASSLSFQLLRRHSRNNLINKFRNPSLPRIHMPRQNIDLKTF...,519,[A0A076FF10],[],...,[],[],[],[],"[PF08417, PF00355]",[],[],[],[],[]
3,A0A076FFM5,"Flavonoid 8-hydroxylase 1, chloroplastic",[1.14.15.-],Ocimum basilicum,39350,"[Eukaryota, Viridiplantae, Streptophyta, Embry...",MPFPMEVLQASSLSFPLLRRHSRNNLINKFRNPTLPRIDIPRQNID...,523,[A0A076FFM5],[],...,[],[],[],[],"[PF08417, PF00355]",[],[],[],[],[]
4,A0A084G2W8,Fungal defensin scedosporisin-2,None,Pseudallescheria apiosperma,563466,"[Eukaryota, Fungi, Dikarya, Ascomycota, Pezizo...",MKFSNISIAALFTILASTAMAAPAADSPDSIVAREPAPVEETYEAP...,94,[A0A084G2W8],[],...,[],[],[],[],[PF01097],[],[],[],[],[]
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13152,V9M2S5,Disease resistance protein RPV1,None,Vitis rotundifolia,103349,"[Eukaryota, Viridiplantae, Streptophyta, Embry...",MASTSSFRASSSSSTPSIPRTTTYDVFLSFRGEDTRYNFTDHLYSA...,1398,[V9M2S5],[],...,[],[],[],[5KU7],"[PF20160, PF23598, PF07725, PF00931, PF01582, ...",[],[],[],[],[]
13153,V9M398,Disease resistance protein RUN1,None,Vitis rotundifolia,103349,"[Eukaryota, Viridiplantae, Streptophyta, Embry...",MASTSSSRASSSSSSSSTPSIPRTITYDVFLSFRGEDTRFNFTDHL...,1331,[V9M398],[],...,[],[],[],"[6O0W, 7RTS, 7RX1, 7S2Z]","[PF20160, PF23598, PF00931, PF01582, PF23282]",[],[],[],[],[]
13154,W4VSH9,Kunitz-type U19-barytoxin-Tl1a,None,Trittame loki,1295018,"[Eukaryota, Metazoa, Ecdysozoa, Arthropoda, Ch...",MNFELIYVSSLLLGICLANQADVVPSDCNLPADAGMCYAYFPMFFY...,176,[W4VSH9],[],...,[],[],[],[],[PF00014],[],[],[],[],[]
13155,W6HZ31,Kunitz-type serine protease inhibitor IrSPI,None,Ixodes ricinus,34613,"[Eukaryota, Metazoa, Ecdysozoa, Arthropoda, Ch...",MKATLVAICFFAAVSYSMGRLTETQCRFPVPVTSCAEGAKLRTVYS...,94,[W6HZ31],[],...,[],[],[],[],[],[],[],[],[],[]
